In [1]:
!pip install tweepy textblob matplotlib

In [5]:
import tweepy
from textblob import TextBlob
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import time

# credentials
consumer_key = 'wKiWEjagqVdAQo4sKvlnutWLr'
consumer_secret = 'gvKcI2hTyRleUJbUDsCLHKEoBsGh9mAdrgz0VIzTNTkw5LtiA3'
access_token = '1783773742294011904-EpvDTDpUcwmGWDtHI8JkBvROW5W1PI'
access_token_secret = 'HemyxIn3mGjcQTFhm824V1894q0u3OLOrppA0mNT1uHdW'

# Authenticate with Twitter using Tweepy
auth = tweepy.OAuth1UserHandler(consumer_key, consumer_secret, access_token, access_token_secret)
api = tweepy.API(auth)

# Specify the Twitter username (without @)
username = 'The_REDWRITER'

# Fetch tweets from the user using the search endpoint
query = f"from:{username}"
tweets = []
for tweet in tweepy.Cursor(api.search_tweets, q=query).items(10):
    tweets.append(tweet)

# Initialize list to store sentiment scores with timestamps
sentiment_scores = []

# Function to fetch direct replies to a tweet
def fetch_replies(tweet):
    replies = []
    query = f"to:{username}"
    for reply in tweepy.Cursor(api.search_tweets, q=query).items():
        if reply.in_reply_to_status_id == tweet.id:
            replies.append(reply)
        if len(replies) >= 50:  # Limit replies to avoid excessive API usage
            break
    return replies

# Iterate over each tweet and fetch its direct replies
for tweet in tweets:
    print(f"Fetching replies for tweet ID: {tweet.id}")
    replies = fetch_replies(tweet)
    print(f"Found {len(replies)} direct replies")
    
    # Perform sentiment analysis on each reply
    for reply in replies:
        analysis = TextBlob(reply.full_text)
        polarity = analysis.sentiment.polarity  # Ranges from -1 to 1
        sentiment_score = (polarity + 1) / 2 * 100  # Convert to 0% to 100%
        timestamp = reply.created_at
        sentiment_scores.append((timestamp, sentiment_score))
    
    # Add a delay to respect rate limits
    time.sleep(1)  # Adjust delay as needed

# Sort sentiment scores by timestamp
sentiment_scores.sort(key=lambda x: x[0])

# Plot the sentiment scores over time
if sentiment_scores:
    timestamps, scores = zip(*sentiment_scores)
    plt.figure(figsize=(10, 5))
    plt.plot(timestamps, scores, marker='o')
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=1))  # Adjust interval as needed
    plt.xlabel('Time')
    plt.ylabel('Sentiment Score (%)')
    plt.title(f'Sentiment Analysis of Replies to Recent Posts by @{username}')
    plt.grid(True)
    plt.show()
else:
    print("No replies found for sentiment analysis.")

Forbidden: 403 Forbidden
453 - You currently have access to a subset of X API V2 endpoints and limited v1.1 endpoints (e.g. media post, oauth) only. If you need access to this endpoint, you may need a different access level. You can learn more here: https://developer.x.com/en/portal/product